<a href="https://colab.research.google.com/github/nikhitarao/nikhitarao_projects/blob/Healthcare-Insurance-Medical-Inflation-Analysis-using-Synthetic-Data/Inpatient_Healthcare_Medical_Procedure_Synthetic_Data_Generation_Member_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Health Insurance Member Database for a Fictional New York based Health Insurer based on Policy Database

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

#### Load the Policy Dataset

In [3]:
from google.colab import files
uploaded = files.upload()  # Upload the file manually
policy_df = pd.read_csv("inpatient_healthcare_insurance_synthetic_policy_dataset.csv")

Saving inpatient_healthcare_insurance_synthetic_policy_dataset.csv to inpatient_healthcare_insurance_synthetic_policy_dataset (1).csv


#### Initialize the Member Dataset

In [4]:
# Initialize an empty DataFrame for Member Dataset
columns = [
    "Policy ID", "Policyholder ID", "Member ID", "Member Ethnicity", "Member Status",
    "Member Marital Status", "Member Designation", "Member Relationship", "Member Gender",
    "Member Age", "Member Income", "Member Chronic Conditions", "Member Smoking Status",
    "Member Alcohol Consumption", "Member Physical Activity Level", "Member Height",
    "Member Weight", "Member BMI Category", "Member Disabilities", "Member Claim History",
    "Member Claim Frequency", "Member Risk Score", "Member Tenure", "Member Productivity Index",
    "Member Work Hours"
]

member_df = pd.DataFrame(columns=columns)

#### Generate Member Data for Each Policy for columns Policy ID, Policyholder ID, Member ID, Member Ethnicity, Member Status, Member Inclusion Date, Member Removal Date

In [ ]:
import pandas as pd
import numpy as np

# Initialize an empty DataFrame for the Member Dataset
member_df = pd.DataFrame(columns=[
    "Policy ID", "Policyholder ID", "Member ID", "Policy Start Date", "Policy End Date",
    "Churn Flag", "Renewal Flag", "Member Ethnicity", "Member Status",
    "Member Inclusion Date", "Member Removal Date"
])

# Helper Functions
def random_date_within_period(start_date, end_date):
    """Generate a random date within the given period."""
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    delta = (end - start).days
    return start + pd.to_timedelta(np.random.randint(0, delta + 1), unit='d')

def assign_ethnicity(location):
    """Assign ethnicity based on location type."""
    ethnicity_distribution = {
        "Urban": {"White": 50, "African-American": 16, "Hispanic": 18, "Asian": 10, "Native American": 0.5, "South Asian": 3, "Other": 2.5},
        "Suburban": {"White": 55, "African-American": 16, "Hispanic": 18, "Asian": 9, "Native American": 1, "South Asian": 0.5, "Other": 0.5},
        "Rural": {"White": 65, "African-American": 14, "Hispanic": 12, "Asian": 5, "Native American": 3, "South Asian": 0.5, "Other": 0.5}
    }
    choices, weights = zip(*ethnicity_distribution[location].items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def determine_member_status(policy_type, member_relationship, member_age):
    """Determine if a member is included or removed in renewals."""
    status = "Included"
    if policy_type == "Family Plan":
        if np.random.rand() < 0.01:  # Deaths
            status = "Removed"
        elif np.random.rand() < 0.02:  # Divorce
            status = "Removed"
        elif member_relationship == "Child" and member_age == 18:  # Age out
            status = "Removed"
    elif policy_type == "Group Health Plan":
        if np.random.rand() < 0.04:  # Employee exits
            status = "Removed"
    return status

def generate_inclusion_and_removal_dates(policy_start, policy_end, policy_type, status, churn_flag):
    """Generate inclusion and removal dates for members."""
    policy_start_date = pd.to_datetime(policy_start)
    policy_end_date = pd.to_datetime(policy_end)
    inclusion_date = random_date_within_period(policy_start_date, policy_end_date)

    if churn_flag == "Churned":
        removal_date = policy_end_date
    elif status == "Removed":
        removal_date = random_date_within_period(inclusion_date, policy_end_date)
    else:
        removal_date = None  # Member is not removed
    return inclusion_date, removal_date


# Ensure 'Number of Members' column is integer and replace 0 with 1
policy_df['Number of Members'] = policy_df['Number of Members'].fillna(1).astype(int)
policy_df.loc[policy_df['Number of Members'] == 0, 'Number of Members'] = 1

# Generate Member Data
all_members = []
for _, policy in policy_df.iterrows():
    policy_id = policy['Policy ID']
    policyholder_id = policy['Policyholder ID']
    num_members = policy['Number of Members']
    location = policy['Location Type']
    policy_type = policy['Policy Type']
    policy_start = policy['Policy Start Date']
    policy_end = policy['Policy End Date']
    churn_flag = policy['Churn Flag']
    renewal_flag = policy['Renewal Flag']

    for member_index in range(1, num_members + 1):
        # Generate Member ID
        member_id = f"{policy_id}-{member_index:08d}"

        # Assign Ethnicity
        ethnicity = assign_ethnicity(location)

        # Determine Member Status
        member_relationship = "Policyholder" if member_index == 1 else "Dependent"
        member_age = np.random.randint(0, 99)  # Example placeholder for age
        status = determine_member_status(policy_type, member_relationship, member_age)

        # Generate Inclusion and Removal Dates
        inclusion_date, removal_date = generate_inclusion_and_removal_dates(
            policy_start, policy_end, policy_type, status, churn_flag
        )

        # Append Member Data
        member = {
            "Policy ID": policy_id,
            "Policyholder ID": policyholder_id,
            "Policy Start Date": policy_start,
            "Policy End Date": policy_end,
            "Churn Flag": churn_flag,
            "Renewal Flag": renewal_flag,
            "Member ID": member_id,
            "Member Ethnicity": ethnicity,
            "Member Status": status,
            "Member Inclusion Date": inclusion_date,
            "Member Removal Date": removal_date
        }
        all_members.append(member)

# Create the Member DataFrame
member_df = pd.DataFrame(all_members)

# Display the first few rows of the member dataset
member_df.head()

In [ ]:
member_df.shape

#### Populate Column Values for columns Member Marital Status, Member Designation, Member Relationship, Member Gender, Member Age, Member Income, Member Chronic Conditions, Member Smoking Status, and Member Alcohol Consumption

In [ ]:
# Helper Functions for New Columns

def assign_marital_status(location):
    """Assign marital status based on location."""
    status_distribution = {
        "Urban": {"Single": 50, "Married": 40, "Divorced": 7, "Widowed": 3},
        "Suburban": {"Single": 45, "Married": 45, "Divorced": 7, "Widowed": 3},
        "Rural": {"Single": 40, "Married": 45, "Divorced": 10, "Widowed": 5}
    }
    choices, weights = zip(*status_distribution[location].items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_designation(policy_type):
    """Assign job roles for Group Health plans."""
    if policy_type == "Group Health Plan":
        choices = ["Executive", "Mid-level Manager", "Entry-level Employee", "Support Staff"]
        weights = [5, 15, 50, 30]
        return np.random.choice(choices, p=np.array(weights) / sum(weights))
    return None  # Not applicable for other plans

def assign_relationship(num_members, member_index):
    """Assign relationship role in the family."""
    if num_members == 1:
        return "Policyholder"
    elif num_members == 2:
        return "Policyholder" if member_index == 1 else "Spouse"
    else:
        roles = ["Policyholder", "Spouse", "Child"]
        return roles[min(member_index - 1, len(roles) - 1)]

def assign_gender(location, is_child):
    """Assign gender based on location and age group."""
    if is_child:
        return np.random.choice(["Male", "Female"], p=[0.5, 0.5])
    gender_distribution = {"Urban": [48.5, 50, 1.5], "Suburban": [49, 50, 1], "Rural": [50, 50, 0]}
    return np.random.choice(["Male", "Female", "Other"], p=np.array(gender_distribution[location]) / 100)

def assign_age(policy_type, relationship):
    """Assign age based on group type and relationship."""
    if policy_type == "Family Plan":
        if relationship == "Policyholder":
            return np.random.randint(30, 50)
        elif relationship == "Spouse":
            return np.random.randint(25, 50)
        elif relationship == "Child":
            return np.random.randint(0, 18)
        elif relationship == "Parent":
            return np.random.randint(50, 70)
    elif policy_type == "Group Health Plan":
        return np.random.randint(22, 65)
    else:
        return np.random.randint(18, 99)

def assign_income(policy_type, designation, location, age):
    """Assign income based on group type, location, and age."""
    if policy_type == "Group Health Plan":
        income_ranges = {
            "Executive": (100000, 200000),
            "Mid-level Manager": (60000, 100000),
            "Entry-level Employee": (40000, 60000),
            "Support Staff": (20000, 40000)
        }
        return np.random.randint(*income_ranges.get(designation, (0, 0)))
    elif policy_type == "Not a Group":
        income_ranges = {"Urban": (50000, 100000), "Suburban": (40000, 80000), "Rural": (30000, 60000)}
        return np.random.randint(*income_ranges[location]) if age > 18 else None
    return None

def assign_chronic_conditions():
    """Assign chronic conditions."""
    conditions = ["None", "Asthma", "Diabetes", "Hypertension", "Depression", "Combination"]
    weights = [60, 5, 8, 12, 15, 5]
    return np.random.choice(conditions, p=np.array(weights) / sum(weights))

def assign_smoking_status():
    """Assign smoking behavior."""
    return np.random.choice(["Smoker", "Non-Smoker", "Ex-Smoker"], p=[0.2, 0.75, 0.05])

def assign_alcohol_consumption():
    """Assign alcohol consumption behavior."""
    return np.random.choice(["None", "Moderate", "High"], p=[0.3, 0.6, 0.1])

# Add New Columns to Member Dataset
updated_members = []
for _, member in member_df.iterrows():
    # Fetch existing data
    location = member["Policyholder ID"]  # Placeholder, map this from policy dataset
    policy_type = member["Policyholder ID"]  # Placeholder, map this from policy dataset
    num_members = member["Policyholder ID"]  # Placeholder, map this from policy dataset
    member_index = int(member["Member ID"].split("-")[-1])

    # Calculate additional fields
    marital_status = assign_marital_status(location)
    designation = assign_designation(policy_type)
    relationship = assign_relationship(num_members, member_index)
    is_child = relationship == "Child"
    gender = assign_gender(location, is_child)
    age = assign_age(policy_type, relationship)
    income = assign_income(policy_type, designation, location, age)
    chronic_conditions = assign_chronic_conditions()
    smoking_status = assign_smoking_status()
    alcohol_consumption = assign_alcohol_consumption()

    # Update member data
    member.update({
        "Member Marital Status": marital_status,
        "Member Designation": designation,
        "Member Relationship": relationship,
        "Member Gender": gender,
        "Member Age": age,
        "Member Income": income,
        "Member Chronic Conditions": chronic_conditions,
        "Member Smoking Status": smoking_status,
        "Member Alcohol Consumption": alcohol_consumption
    })
    updated_members.append(member)

# Update the DataFrame
member_df = pd.DataFrame(updated_members)

member_df.head()

In [ ]:
member_df.shape

#### Populate Column Values for columns Member Physical Activity Level, Member Height, Member Weight, Member BMI Category, Member Disabilities, Member Claim History, Member Claim Frequency, Member Risk Score, Member Tenure, Member Productivity Index, Member Work Hours



In [ ]:
# Helper Functions for New Columns

def assign_physical_activity_level():
    """Assign physical activity level."""
    return np.random.choice(["Sedentary", "Light", "Moderate", "High"], p=[0.3, 0.4, 0.2, 0.1])

def assign_height(gender, age):
    """Assign height based on gender and age."""
    if age <= 18:  # Child
        return np.random.randint(100, 170)  # Height in cm
    if gender == "Male":
        return np.random.randint(160, 200)
    elif gender == "Female":
        return np.random.randint(150, 180)
    return np.random.randint(150, 190)  # Default for "Other"

def assign_weight(height, age, chronic_conditions):
    """Assign weight based on height, age, and chronic conditions."""
    bmi = np.random.normal(22, 3)  # Normal BMI with some variability
    if "Hypertension" in chronic_conditions or "Diabetes" in chronic_conditions:
        bmi += np.random.uniform(2, 5)
    if age > 50:
        bmi += np.random.uniform(1, 3)
    weight = (bmi * (height / 100) ** 2)  # Calculate weight from BMI and height
    return int(weight)

def calculate_bmi_category(weight, height):
    """Calculate BMI category."""
    bmi = weight / (height / 100) ** 2
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 24.9:
        return "Normal"
    elif 25 <= bmi < 29.9:
        return "Overweight"
    else:
        return "Obese"

def assign_disabilities(age, chronic_conditions, industry=None):
    """Assign disabilities based on age, chronic conditions, and industry."""
    base_probabilities = {"None": 90, "Physical": 5, "Cognitive": 3, "Both": 2}
    if age > 50:
        base_probabilities["Physical"] += 3
        base_probabilities["Both"] += 2
    if industry == "Manufacturing":
        base_probabilities["Physical"] += 2
    elif industry in ["Finance", "Tech"]:
        base_probabilities["Cognitive"] += 1
    if "Hypertension" in chronic_conditions or "Diabetes" in chronic_conditions:
        base_probabilities["Both"] += 3
    choices, weights = zip(*base_probabilities.items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_claim_history(age, chronic_conditions):
    """Assign claim history."""
    base_probabilities = {"No Claims": 80, "Minor Claims": 15, "Major Claims": 5}
    if age > 50:
        base_probabilities["Minor Claims"] += 5
        base_probabilities["Major Claims"] += 2
    if "Diabetes" in chronic_conditions or "Hypertension" in chronic_conditions:
        base_probabilities["Minor Claims"] += 7
    if "Combination" in chronic_conditions:
        base_probabilities["Major Claims"] += 10
    choices, weights = zip(*base_probabilities.items())
    return np.random.choice(choices, p=np.array(weights) / sum(weights))

def assign_claim_frequency(claim_history, chronic_conditions):
    """Assign claim frequency based on history and conditions."""
    if claim_history == "No Claims":
        return 0
    elif claim_history == "Minor Claims":
        return np.random.choice([1, 2], p=[0.8, 0.2])
    elif claim_history == "Major Claims":
        return np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])

def calculate_risk_score(age, chronic_conditions, claim_history, smoking_status, physical_activity_level, alcohol_consumption):
    """Calculate risk score."""
    score = 0
    score += (age - 30) // 5
    score += 5 * chronic_conditions.count("None") if chronic_conditions != "None" else 0
    score += 3 if claim_history == "Minor Claims" else 5 if claim_history == "Major Claims" else 0
    score += 5 if smoking_status == "Smoker" else 0
    score += 3 if physical_activity_level == "Sedentary" else 0
    score += 2 if alcohol_consumption == "High" else 0
    return score

# Add New Columns to Member Dataset
updated_members = []
for _, member in member_df.iterrows():
    # Fetch existing data
    gender = member["Member Gender"]
    age = member["Member Age"]
    chronic_conditions = member["Member Chronic Conditions"]
    industry = member.get("Industry Type", None)

    # Calculate additional fields
    physical_activity_level = assign_physical_activity_level()
    height = assign_height(gender, age)
    weight = assign_weight(height, age, chronic_conditions)
    bmi_category = calculate_bmi_category(weight, height)
    disabilities = assign_disabilities(age, chronic_conditions, industry)
    claim_history = assign_claim_history(age, chronic_conditions)
    claim_frequency = assign_claim_frequency(claim_history, chronic_conditions)
    risk_score = calculate_risk_score(age, chronic_conditions, claim_history, member["Member Smoking Status"], physical_activity_level, member["Member Alcohol Consumption"])

    # Update member data
    member.update({
        "Member Physical Activity Level": physical_activity_level,
        "Member Height": height,
        "Member Weight": weight,
        "Member BMI Category": bmi_category,
        "Member Disabilities": disabilities,
        "Member Claim History": claim_history,
        "Member Claim Frequency": claim_frequency,
        "Member Risk Score": risk_score
    })
    updated_members.append(member)

# Update the DataFrame
member_df = pd.DataFrame(updated_members)

member_df.head()

In [ ]:
member_df.shape

#### Writing the Synthetic Policy Dataset to a CSV File and then Downloaded


In [ ]:
# Save the dataset to a local file in Colab
file_path = 'inpatient_healthcare_insurance_synthetic_member_dataset.csv'
member_df.to_csv(file_path, index=False)
print(f"Dataset saved temporarily at {file_path}")

In [ ]:
from google.colab import files

# Download the file
files.download(file_path)